# NB05 - Multimodal U-Net M3 (S1 VV/VH + DEM + rain t-3/t-2/t-1)

Controlled comparison against locked NB04 (S1-only M0: test IoU 0.6684 / F1 0.8012). **Only intended change: 6 input channels instead of 2** (first conv 6→32; +1152 params). Same splits, seed 42, train-derived norm stats, loaders, UNet-32 family, masked 0.5BCE+0.5Dice, AdamW 1e-3/1e-4, plateau scheduler, patience-8 val-IoU selection, thr 0.5, masking. No tuning to beat NB04 - whatever the evidence says stands.

NB05 = M3 (full multimodal). NB06 will do the M0/M1/M2/M3 ablations. Bolivia untouched (NB07).

## 1. Load project context

In [ ]:
from pathlib import Path
CTX = Path(r"C:/Users/Swarnim/Desktop/ML projects/tsao/project_context")
print("\n".join((CTX / "NEXT_STEPS.md").read_text(encoding="utf-8").splitlines()[:11]))
import json
nb04 = json.loads((Path(r"C:/Users/Swarnim/Desktop/ML projects/tsao/results") / "nb04_results.json").read_text())
print("\nLOCKED NB04 baseline test:", {k: round(nb04["test"][k], 4) for k in ("iou", "f1", "precision", "recall")})
print("NB04 val:", {k: round(nb04["best_val"][k], 4) for k in ("iou", "f1", "precision", "recall")}, "ep", nb04["best_epoch"])


## 2. Imports / configuration (identical to NB04 except input channels)

In [ ]:
from pathlib import Path
import json, random, time, datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
PROJECT_ROOT = Path(r"C:/Users/Swarnim/Desktop/ML projects/tsao")
PROCESSED_ROOT = PROJECT_ROOT / "processed"
MODEL_DIR = PROJECT_ROOT / "models"; RESULT_DIR = PROJECT_ROOT / "results"
BEST_CKPT = MODEL_DIR / "nb05_multimodal_unet_best.pt"
CONFIG_JSON = MODEL_DIR / "nb05_multimodal_unet_config.json"
HIST_CSV = RESULT_DIR / "nb05_training_history.csv"
RESULTS_JSON = RESULT_DIR / "nb05_results.json"
CMP_JSON = RESULT_DIR / "nb04_vs_nb05_comparison.json"
NB04_CKPT = MODEL_DIR / "nb04_s1_unet_best.pt"  # read-only, for qualitative comparison
SEED = 42; BATCH_SIZE = 4; LR = 1e-3; WEIGHT_DECAY = 1e-4; EPOCHS = 40; PATIENCE = 8
BASE_CH = 32; IN_CH = 6  # THE experimental change: all six NB02 channels, original order
LOSS_W = {"bce": 0.5, "dice": 0.5}; THR = 0.5; EPS = 1e-6
USE_AMP = torch.cuda.is_available()
EXPECTED_CH = ["VV_dB", "VH_dB", "DEM_m", "rain_t-3", "rain_t-2", "rain_t-1"]
print(f"IN_CH={IN_CH} epochs={EPOCHS} batch={BATCH_SIZE} lr={LR} wd={WEIGHT_DECAY} patience={PATIENCE} amp={USE_AMP}")


## 3. Reproducibility / device

In [ ]:
import random
import numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True  # same as NB04 (documented)
print(f"device={DEVICE} | torch={torch.__version__} | gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
g = torch.Generator().manual_seed(SEED)


## 4. Load NB03 Dataset / DataLoaders (verbatim logic - full 6 channels, original order)

In [ ]:
import json
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader
st = json.loads((PROCESSED_ROOT / "normalization_stats.json").read_text())
assert st["split"] == "train" and st["channels"] == EXPECTED_CH, st["channels"]
print("train stats channels (order preserved):", st["channels"])

class FloodDataset(Dataset):  # identical to NB03 (locked)
    def __init__(self, split, processed_root=PROCESSED_ROOT, stats_path=PROCESSED_ROOT / "normalization_stats.json"):
        assert split in ("train", "valid", "test", "bolivia"), split
        s = json.loads(stats_path.read_text())
        assert s["split"] == "train"
        self.mean = np.array(s["mean"], dtype=np.float32).reshape(-1, 1, 1)
        self.std = np.array(s["std"], dtype=np.float32).reshape(-1, 1, 1)
        assert (self.std > 0).all()
        self.paths = sorted((processed_root / split).glob("*.npz"))
        assert len(self.paths) > 0
        self.split = split
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        d = np.load(self.paths[i], allow_pickle=True)
        x = (d["X"].astype(np.float32) - self.mean) / self.std
        x = np.where(np.isfinite(x), x, 0.0).astype(np.float32)
        return {"x": torch.from_numpy(x), "y": torch.from_numpy(d["y"].astype(np.float32)),
                "mask": torch.from_numpy(d["valid_mask"].astype(bool)), "chip_id": str(d["chip_id"])}

ds = {s: FloodDataset(s) for s in ("train", "valid", "test", "bolivia")}
assert {s: len(ds[s]) for s in ds} == {"train": 252, "valid": 89, "test": 90, "bolivia": 15}
loaders = {
    "train": DataLoader(ds["train"], batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available(), generator=g),
    "valid": DataLoader(ds["valid"], batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available()),
    "test": DataLoader(ds["test"], batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available()),
    "bolivia": DataLoader(ds["bolivia"], batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available()),
}
print({s: len(ds[s]) for s in ds}, "| splits frozen, stats=train-derived")


## 5. Six-channel input verification (the experimental change, gated explicitly)

In [ ]:
b = next(iter(loaders["train"]))
assert tuple(b["x"].shape) == (BATCH_SIZE, 6, 512, 512), b["x"].shape
assert bool(torch.isfinite(b["x"]).all())
print(f"MODEL INPUT VERIFIED: {tuple(b['x'].shape)} = [VV, VH, DEM, rain t-3/t-2/t-1] in NB02 order. No S2, no event-day rain.")
del b


## 6. U-Net architecture (NB04 family; only first conv 6→32)

In [ ]:
import torch.nn as nn
class DoubleConv(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(ci, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
                                 nn.Conv2d(co, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):  # identical to NB04 except in_ch
    def __init__(self, in_ch=6, base=32):
        super().__init__()
        self.e1 = DoubleConv(in_ch, base);      self.p1 = nn.MaxPool2d(2)
        self.e2 = DoubleConv(base, base * 2);   self.p2 = nn.MaxPool2d(2)
        self.e3 = DoubleConv(base * 2, base * 4); self.p3 = nn.MaxPool2d(2)
        self.e4 = DoubleConv(base * 4, base * 8); self.p4 = nn.MaxPool2d(2)
        self.neck = DoubleConv(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, 2, 2);  self.d4 = DoubleConv(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2);   self.d3 = DoubleConv(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2);    self.d2 = DoubleConv(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2);        self.d1 = DoubleConv(base * 2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        x1 = self.e1(x); x2 = self.e2(self.p1(x1)); x3 = self.e3(self.p2(x2)); x4 = self.e4(self.p3(x3))
        n = self.neck(self.p4(x4))
        m = self.d4(torch.cat([self.u4(n), x4], 1)); m = self.d3(torch.cat([self.u3(m), x3], 1))
        m = self.d2(torch.cat([self.u2(m), x2], 1)); m = self.d1(torch.cat([self.u1(m), x1], 1))
        return self.out(m)

model = UNet(IN_CH, BASE_CH).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"UNet-32 multimodal params: {n_params:,} (NB04 S1-only: 7,762,753; delta={n_params - 7762753:+d}, first-conv only)")
assert n_params - 7762753 == (6 - 2) * 32 * 9, "param delta must equal first-conv change only"
x0 = torch.randn(1, 6, 512, 512, device=DEVICE)
with torch.no_grad(): y0 = model(x0)
assert tuple(y0.shape) == (1, 1, 512, 512)
print("forward OK", tuple(y0.shape))
del x0, y0


## 7. Masked BCE + Dice (same formula as NB04)

In [ ]:
import torch.nn.functional as F
def masked_bce_dice(logits, target, mask, w_bce=0.5, w_dice=0.5, eps=1e-6):
    m = mask.float()
    n = m.sum().clamp_min(1.0)
    bce = (F.binary_cross_entropy_with_logits(logits, target, reduction="none") * m).sum() / n
    p = torch.sigmoid(logits)
    inter = (p * target * m).sum()
    dice = 1.0 - (2.0 * inter + eps) / ((p * m).sum() + (target * m).sum() + eps)
    return w_bce * bce + w_dice * dice, {"bce": bce.detach(), "dice": dice.detach()}
print("Loss = 0.5*BCE + 0.5*Dice over valid_mask only (logits in, zero invalid contribution).")


## 8. Metrics (same aggregate definitions as NB04)

In [ ]:
@torch.no_grad()
def accumulate_metrics(logits, target, mask, acc, thr=0.5, eps=1e-9):
    pred = (torch.sigmoid(logits) >= thr).float()
    t, m = target.float(), mask.float()
    acc["tp"] += float(((pred == 1) & (t == 1) & (m == 1)).sum())
    acc["fp"] += float(((pred == 1) & (t == 0) & (m == 1)).sum())
    acc["fn"] += float(((pred == 0) & (t == 1) & (m == 1)).sum())
    acc["tn"] += float(((pred == 0) & (t == 0) & (m == 1)).sum())

def summarize(acc, eps=1e-9):
    tp, fp, fn, tn = (acc[k] for k in ("tp", "fp", "fn", "tn"))
    iou = tp / (tp + fp + fn + eps)
    prec = tp / (tp + fp + eps); rec = tp / (tp + fn + eps)
    f1 = 2 * prec * rec / (prec + rec + eps)
    return {"iou": iou, "f1": f1, "precision": prec, "recall": rec, "tp": tp, "fp": fp, "fn": fn, "tn": tn}


## 9. Sanity checks (6ch batch → forward → masked loss → step, all finite)

In [ ]:
model.train()
opt0 = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
b = next(iter(loaders["train"]))
xb, yb, mb = b["x"].to(DEVICE), b["y"].unsqueeze(1).to(DEVICE), b["mask"].unsqueeze(1).to(DEVICE)
assert xb.shape[1] == 6, "multimodal model must see exactly 6 channels"
opt0.zero_grad()
out = model(xb)
assert tuple(out.shape) == (BATCH_SIZE, 1, 512, 512)
loss, parts = masked_bce_dice(out, yb, mb, LOSS_W["bce"], LOSS_W["dice"])
loss.backward()
assert torch.isfinite(loss) and torch.isfinite(out).all()
assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
opt0.step()
print(f"sanity: fwd {tuple(out.shape)} loss={float(loss):.4f} bwd+step OK, all finite")
del b, xb, yb, mb, out, loss


## 10. Tiny overfit test (fresh model, 1 batch of 2 chips, 60 steps, must reach <60% of init)

In [ ]:
torch.manual_seed(SEED)
probe = UNet(IN_CH, BASE_CH).to(DEVICE).train()
popt = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=0.0)
bt = next(iter(loaders["train"]))
px, py, pm = bt["x"][:2].to(DEVICE), bt["y"][:2].unsqueeze(1).to(DEVICE), bt["mask"][:2].unsqueeze(1).to(DEVICE)
hist = []
for s in range(60):
    popt.zero_grad()
    l, _ = masked_bce_dice(probe(px), py, pm)
    l.backward(); popt.step()
    hist.append(float(l))
print(f"overfit: {hist[0]:.4f} -> {hist[-1]:.4f} (x{hist[0] / max(hist[-1], 1e-9):.1f})")
assert hist[-1] < 0.6 * hist[0], "cannot overfit - STOP"
print("OVERFIT GATE PASS")
del probe, popt, bt, px, py, pm
torch.manual_seed(SEED)
model = UNet(IN_CH, BASE_CH).to(DEVICE)


## 11. Masked-loss invariance test (repeat of NB04 gate)

In [ ]:
torch.manual_seed(0)
Lt, Tt = torch.randn(2, 1, 32, 32), (torch.rand(2, 1, 32, 32) > 0.9).float()
Mt = torch.ones(2, 1, 32, 32); Mt[:, :, :8, :8] = 0
l1, _ = masked_bce_dice(Lt, Tt, Mt)
Lt2, Tt2 = Lt.clone(), Tt.clone()
Lt2[~Mt.bool()] = 999.0; Tt2[~Mt.bool()] = 1.0 - Tt2[~Mt.bool()]
l2, _ = masked_bce_dice(Lt2, Tt2, Mt)
assert l1.item() == l2.item(), (l1.item(), l2.item())
print(f"masked-loss invariance PASS ({float(l1):.6f} unchanged by invalid-pixel corruption)")


## 12. Training loop (same policy as NB04: AdamW, plateau on val loss, AMP, early stop on val IoU p=8)

In [ ]:
import time
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=4)
scaler = torch.amp.GradScaler("cuda") if USE_AMP else None
history, best_iou, bad, t_start = [], -1.0, 0, time.time()

@torch.no_grad()
def evaluate(loader_split, net):
    net.eval()
    acc = {"tp": 0.0, "fp": 0.0, "fn": 0.0, "tn": 0.0}
    tot, n = 0.0, 0.0
    for b in loaders[loader_split]:
        x = b["x"].to(DEVICE, non_blocking=True)
        y = b["y"].unsqueeze(1).to(DEVICE, non_blocking=True)
        m = b["mask"].unsqueeze(1).to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            out = net(x)
            l, _ = masked_bce_dice(out, y, m)
        tot += float(l) * x.shape[0]; n += x.shape[0]
        accumulate_metrics(out.float(), y, m, acc, THR)
    return {"loss": tot / max(n, 1), **summarize(acc)}

for ep in range(1, EPOCHS + 1):
    model.train()
    tl, n = 0.0, 0
    for b in loaders["train"]:
        x = b["x"].to(DEVICE, non_blocking=True)
        y = b["y"].unsqueeze(1).to(DEVICE, non_blocking=True)
        m = b["mask"].unsqueeze(1).to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            out = model(x)
            l, _ = masked_bce_dice(out, y, m, LOSS_W["bce"], LOSS_W["dice"])
        assert torch.isfinite(l), f"non-finite train loss at epoch {ep}"
        if scaler: scaler.scale(l).backward(); scaler.step(opt); scaler.update()
        else: l.backward(); opt.step()
        tl += float(l) * x.shape[0]; n += x.shape[0]
    vm = evaluate("valid", model)
    lr_now = opt.param_groups[0]["lr"]
    sched.step(vm["loss"])
    history.append({"epoch": ep, "train_loss": tl / n, "val_loss": vm["loss"], "val_iou": vm["iou"],
                    "val_f1": vm["f1"], "val_precision": vm["precision"], "val_recall": vm["recall"], "lr": lr_now})
    improved = vm["iou"] > best_iou
    if improved:
        best_iou, bad = vm["iou"], 0
        torch.save({"state_dict": model.state_dict(), "epoch": ep, "val": vm}, BEST_CKPT)
    else: bad += 1
    print(f"ep {ep:02d} train={tl/n:.4f} val_loss={vm['loss']:.4f} iou={vm['iou']:.4f} f1={vm['f1']:.4f} lr={lr_now:.1e} {'*BEST*' if improved else f'(bad {bad})'}", flush=True)
    if bad >= PATIENCE:
        print(f"early stopping at epoch {ep}"); break
train_time_s = time.time() - t_start
pd.DataFrame(history).to_csv(HIST_CSV, index=False)
print(f"\nTRAIN DONE: {len(history)} epochs, {train_time_s/60:.1f} min, best val IoU={best_iou:.4f}")


## 13. Validation (best-checkpoint recap; test still unseen)

In [ ]:
ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
print(f"best epoch: {ckpt['epoch']} | " + ", ".join(f"{k}={v:.4f}" for k, v in ckpt["val"].items() if isinstance(v, float)))
print(f"epochs run: {len(history)} / {EPOCHS}")


## 14. Best checkpoint selection + config snapshot

In [ ]:
cfg = {"model": "UNet", "in_channels": IN_CH, "channels": EXPECTED_CH, "base_channels": BASE_CH, "params": n_params,
       "arch": "6-32-64-128-256-[512]-256-128-64-32-1 (NB04 family, first conv only differs)",
       "normalization": "NB03 train-derived stats (unchanged file)", "loss": "0.5*maskedBCE + 0.5*maskedDice", "threshold": THR,
       "optimizer": "AdamW", "lr": LR, "weight_decay": WEIGHT_DECAY, "scheduler": "ReduceLROnPlateau(min,val_loss,f=0.5,p=4)",
       "batch_size": BATCH_SIZE, "epochs_run": len(history), "selection": "best validation IoU",
       "best_epoch": ckpt["epoch"], "best_val": ckpt["val"], "seed": SEED, "amp": USE_AMP, "device": str(DEVICE),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
       "torch": torch.__version__, "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}
CONFIG_JSON.write_text(json.dumps(cfg, indent=2))
print(f"config saved; best ep {cfg['best_epoch']} val IoU={cfg['best_val']['iou']:.4f}")


## 15. Training curves

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(history)
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
ax[0, 0].plot(h["epoch"], h["train_loss"], label="train"); ax[0, 0].plot(h["epoch"], h["val_loss"], label="valid")
ax[0, 0].set_title("loss (masked BCE+Dice)"); ax[0, 0].legend()
ax[0, 1].plot(h["epoch"], h["val_iou"], label="IoU"); ax[0, 1].plot(h["epoch"], h["val_f1"], label="F1")
ax[0, 1].set_title("validation IoU/F1"); ax[0, 1].legend()
ax[1, 0].plot(h["epoch"], h["val_precision"], label="prec"); ax[1, 0].plot(h["epoch"], h["val_recall"], label="rec")
ax[1, 0].set_title("validation precision/recall"); ax[1, 0].legend()
ax[1, 1].plot(h["epoch"], h["lr"]); ax[1, 1].set_title("learning rate"); ax[1, 1].set_yscale("log")
for a in ax.ravel(): a.set_xlabel("epoch")
fig.suptitle(f"NB05 multimodal UNet training (best val IoU={best_iou:.4f} @ep {ckpt['epoch']})")
fig.tight_layout(); plt.show()


## 16. Final test evaluation (BEST val checkpoint, exactly once)

In [ ]:
assert len(ds["test"]) == 90 and len(ds["bolivia"]) == 15
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)["state_dict"])
test_m = evaluate("test", model)
print("TEST (best-val checkpoint, single evaluation):")
print(", ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v:.0f}" for k, v in test_m.items()))
bb = next(iter(loaders["bolivia"]))
assert tuple(bb["x"].shape[1:]) == (6, 512, 512)
print(f"Bolivia loader smoke OK {tuple(bb['x'].shape)} - no metrics (NB07 territory)")
del bb


## 17. NB04 vs NB05 comparison
Locked NB04 numbers are read from `results/nb04_results.json` (never recomputed/edited). Deltas = NB05 − NB04. No superiority claim unless evidence supports it.

In [ ]:
rows = []
for scope, a, b in [("valid", nb04["best_val"], ckpt["val"]), ("test", nb04["test"], test_m)]:
    for m in ("iou", "f1", "precision", "recall"):
        rows.append({"scope": scope, "metric": m, "NB04_S1": round(a[m], 4), "NB05_multi": round(b[m], 4), "delta": round(b[m] - a[m], 4)})
cmp_df = pd.DataFrame(rows)
display(cmp_df)
d_iou, d_f1 = test_m["iou"] - nb04["test"]["iou"], test_m["f1"] - nb04["test"]["f1"]
print(f"\nTest ΔIoU={d_iou:+.4f} ΔF1={d_f1:+.4f}")
if d_iou > 0.005: print("Evidence: multimodal input improves test IoU over the S1-only baseline.")
elif d_iou < -0.005: print("Evidence: multimodal input does NOT improve test IoU here (valid scientific result).")
else: print("Evidence: test IoU essentially tied (|Δ|<=0.005); environmental channels show no clear added value at this operating point.")


## 18. Qualitative comparison (SAME chips, SAME threshold)
Same 5 chips as NB04 §16 (deterministic valid_frac>0 selection): NB04 binary vs NB05 binary vs GT, plus a disagreement map (green=agree flood, white=agree background, red=NB05-only, blue=NB04-only, grey=invalid). No cherry-picking.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
m4 = UNet(2, BASE_CH).to(DEVICE).eval()  # locked NB04 weights, read-only use
m4.load_state_dict(torch.load(NB04_CKPT, map_location=DEVICE, weights_only=False)["state_dict"])
model.eval()
mp = pd.read_csv(PROCESSED_ROOT / "metadata.csv")
def _shown(split, idxs):
    ok = mp[(mp["split"] == split) & (mp["valid_frac"] > 0)].sort_values("chip_id")["chip_id"].tolist()
    return [(split, ok[i]) for i in idxs]
shown = _shown("valid", (0, 89 // 3, 2 * 89 // 3)) + _shown("test", (0, 89 // 2))
print("chips:", shown)
dis_cmap = ListedColormap(["#ffffff", "#00c000", "#ff0000", "#0060ff", "#808080"])  # agree-bg, agree-flood, M3-only, M0-only, invalid
mn = np.array(st["mean"], dtype=np.float32).reshape(-1, 1, 1); sd = np.array(st["std"], dtype=np.float32).reshape(-1, 1, 1)
for split, stem in shown:
    raw = np.load(PROCESSED_ROOT / split / (stem + ".npz"), allow_pickle=True)
    xn = ((raw["X"].astype(np.float32) - mn) / sd)
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=USE_AMP):
        p4 = torch.sigmoid(m4(torch.from_numpy(xn[:2]).unsqueeze(0).to(DEVICE)))[0, 0].cpu().numpy()
        p5 = torch.sigmoid(model(torch.from_numpy(xn).unsqueeze(0).to(DEVICE)))[0, 0].cpu().numpy()
    y = raw["y"].astype(np.float32); m = raw["valid_mask"].astype(bool)
    b4, b5 = (p4 >= THR).astype(np.uint8), (p5 >= THR).astype(np.uint8)
    dis = np.full_like(y, 0, dtype=np.uint8); dis[(b4==1)&(b5==1)&m]=1; dis[(b5==1)&(b4==0)&m]=2; dis[(b4==1)&(b5==0)&m]=3; dis[~m]=4
    fig, ax = plt.subplots(1, 6, figsize=(17, 3.2))
    ax[0].imshow(raw["X"][0], cmap="gray"); ax[0].set_title(f"VV {stem}")
    ax[1].imshow(raw["X"][1], cmap="gray"); ax[1].set_title("VH")
    ax[2].imshow(np.where(m, y, -1), cmap=ListedColormap(["#808080","#1a1a1a","#00b3ff"]), vmin=-1, vmax=1, interpolation="nearest")
    ax[2].set_title(f"GT flood={(y[m]==1).mean()*100:.1f}%")
    ax[3].imshow(b4, cmap="gray", vmin=0, vmax=1, interpolation="nearest"); ax[3].set_title("NB04 M0")
    ax[4].imshow(b5, cmap="gray", vmin=0, vmax=1, interpolation="nearest"); ax[4].set_title("NB05 M3")
    ax[5].imshow(dis, cmap=dis_cmap, vmin=0, vmax=4, interpolation="nearest"); ax[5].set_title("red=M3-only blue=M0-only")
    for a in ax: a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"{split}: {stem} (same chips/thr for M0 vs M3)")
    fig.tight_layout(); plt.show()
    del xn, p4, p5, y, m, b4, b5, dis
print("Side-by-side panels above (grey=invalid in GT/disagreement).")


## 19. Final results (NB05 artifacts + locked comparison file)

In [ ]:
results = {"notebook": "NB05_Multimodal_UNet", "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
           "input": EXPECTED_CH, "in_shape": [6, 512, 512], "model": "UNet-32 (NB04 family)", "params": n_params,
           "loss": "0.5*maskedBCE+0.5*maskedDice", "config": {"seed": SEED, "batch": BATCH_SIZE, "lr": LR, "wd": WEIGHT_DECAY, "epochs_max": EPOCHS, "patience": PATIENCE},
           "train_time_min": round(train_time_s / 60, 2), "epochs_run": len(history), "best_epoch": ckpt["epoch"],
           "best_val": ckpt["val"], "test": test_m, "threshold": THR, "device": str(DEVICE),
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "amp": USE_AMP}
RESULTS_JSON.write_text(json.dumps(results, indent=2, default=float))
cmp = {"nb04_locked_test": {k: nb04["test"][k] for k in ("iou", "f1", "precision", "recall")},
       "nb05_test": {k: test_m[k] for k in ("iou", "f1", "precision", "recall")},
       "delta_test": {k: round(test_m[k] - nb04["test"][k], 4) for k in ("iou", "f1", "precision", "recall")},
       "nb04_locked_val": {k: nb04["best_val"][k] for k in ("iou", "f1", "precision", "recall")},
       "nb05_val": {k: ckpt["val"][k] for k in ("iou", "f1", "precision", "recall")},
       "created_utc": results["created_utc"]}
CMP_JSON.write_text(json.dumps(cmp, indent=2))
print(json.dumps({"nb05_test_iou": round(test_m["iou"], 4), "delta_iou": cmp["delta_test"]["iou"], "epochs": len(history)}, indent=2))


## 20. Project context update

In [ ]:
import datetime, re
now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
di = round(test_m["iou"] - nb04["test"]["iou"], 4); df = round(test_m["f1"] - nb04["test"]["f1"], 4)
entry = (f"\n## {now} - NB05_Multimodal_UNet COMPLETE\n"
         f"- UNet-32 early-fusion 6ch ({n_params:,} params, +1152 vs NB04); same seed/loss/opt/sched/patience/thr as NB04\n"
         f"- {len(history)} epochs, {results['train_time_min']} min; best ep {ckpt['epoch']} val IoU={ckpt['val']['iou']:.4f} F1={ckpt['val']['f1']:.4f}\n"
         f"- TEST (once): IoU={test_m['iou']:.4f} F1={test_m['f1']:.4f} P={test_m['precision']:.4f} R={test_m['recall']:.4f}\n"
         f"- vs NB04 locked (0.6684/0.8012): dIoU={di:+.4f} dF1={df:+.4f} (see results/nb04_vs_nb05_comparison.json)\n"
         f"- Gates (sanity/overfit/mask-invariance) PASS. Bolivia untouched. NB05 safe to lock; next NB06 ablations M0/M1/M2/M3.\n")
(CTX / "CHANGELOG.md").write_text((CTX / "CHANGELOG.md").read_text(encoding="utf-8") + entry, encoding="utf-8")
ps = CTX / "PROJECT_STATE.md"
t = ps.read_text(encoding="utf-8")
t = re.sub(r"^\*\*Last updated:\*\*.*$", f"**Last updated:** {now} (NB05 complete: yes)", t, count=1, flags=re.M)
t += f"\n\n## NB05 result ({now})\n- Multimodal M3: val IoU={ckpt['val']['iou']:.4f}, test IoU={test_m['iou']:.4f} (d={di:+.4f} vs NB04).\n"
ps.write_text(t, encoding="utf-8")
dc = CTX / "DECISIONS.md"
t = dc.read_text(encoding="utf-8")
t = re.sub(r"^\*\*Last updated:\*\*.*$", f"**Last updated:** {now} (D13 M3)", t, count=1, flags=re.M)
t += "\n## D13 - NB05 early-fusion M3 comparison (" + now + ")\nControlled NB04 replica, only first conv 6ch; M3 established for NB06 ablations; verdict from evidence only.\n"
dc.write_text(t, encoding="utf-8")
nx = CTX / "NEXT_STEPS.md"
t = nx.read_text(encoding="utf-8")
t = re.sub(r"^\*\*Last updated:\*\*.*$", f"**Last updated:** {now}", t, count=1, flags=re.M)
t += f"\n\n## Update ({now})\n- NB05 COMPLETE. Next: NB06 controlled ablations M0/M1/M2/M3.\n"
nx.write_text(t, encoding="utf-8")
print(entry)
print("Context files updated:", now)
